# SVM : exploration

Ce notebook teste un SVM à noyau RBF avec les mêmes règles que les autres modèles : découpage commun entre entraînement et test (le jeu de test n'est pas utilisé ici) validation croisée stratifiée à 5 plis sur le jeu d'entraînement et PR-AUC comme métrique principale.
Ici aussi je vais essayer de faire en sorte que chaque essai soit enregistré dans MLflow. 

Nb : comme on est sur un SVM c'est surtout les métriques d'AUC qu'on va comparer avec les autres modèles.

## Importation dataset + packages

In [2]:
import mlflow
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

from base_commune.outils import (
    RANDOM_STATE,
    charger_donnees,
    configurer_mlflow,
    decouper_donnees,
    evaluer_modele,
    exporter_runs,
    tags_reproductibilite,
)

lancer_les_essais = True  # Je reprend les conseils de l'ia sur mon RF pour éviter d'avoir le même pb

In [4]:
donnees_brutes = charger_donnees()
x_train, x_test, y_train, y_test = decouper_donnees(donnees_brutes)
print("Entraînement :", x_train.shape)
print("Test (pas utilisé ici juste pr verif) :", x_test.shape)

Entraînement : (8000, 10)
Test (pas utilisé ici juste pr verif) : (2000, 10)


In [ ]:
var_num = ["CreditScore", "Tenure", "Balance", "EstimatedSalary", "Age"]
var_cate = ["Geography", "Gender", "NumOfProducts"]
var_binaires = ["HasCrCard", "IsActiveMember"]

## Construction modèle + suivi MlFlow

In [6]:
def construire_modele_svm(valeur_C=1.0, valeur_gamma="scale", ponderation=None):
    pretraitement = ColumnTransformer(
        transformers=[
            ("variables_numeriques", StandardScaler(), var_num),
            ("variables_categorielles", OneHotEncoder(handle_unknown="ignore"), var_cate),
            ("variables_binaires", "passthrough", var_binaires),],remainder="drop",)
    modele_svm = SVC(kernel="rbf", C=valeur_C, gamma=valeur_gamma,class_weight=ponderation)
    return Pipeline([("pretraitement", pretraitement), ("svm", modele_svm)])

In [ ]:
configurer_mlflow("churn_bancaire")
tt_cols = var_binaires + var_cate + var_num

def enregistrer_un_essai(modele, nom_essai, hyperparametres):
    with mlflow.start_run(run_name=nom_essai):
        mlflow.set_tags({"model_family": "svm", "author": "roman", "stage": "exploration"})
        mlflow.set_tags(tags_reproductibilite())

        mlflow.log_params(hyperparametres)
        mlflow.log_params({
            "features": ",".join(tt_cols),
            "cv_folds": 5,
            "random_state": RANDOM_STATE,
            "test_size": 0.2,
            "n_train": len(x_train),
        })

        metriques = evaluer_modele(modele, x_train, y_train) # Rappel jfais ma cross validation via la fonction de outils.py
        mlflow.log_metrics(metriques)
    return metriques

## Test de différentes config
N'étant pas un expert des méthodes type SVM je vais tester les différents hyper-paras de manière un peu arbitraire, j'ai conscience qu'on peut surement largement optimiser ça mais ce n'est pas grave le but ici encore une fois c'est d'avoir des bonnes pratiques MLops + que le modèle en tant que tel

In [9]:
if lancer_les_essais: # Le if pour pas avoir le même problème que cque j'avais eu avec le RF, je trv ce conseil ce l'ia pertinent pour le coup 
    for valeur_C in [0.1, 1, 10]:
        for valeur_gamma in ["scale", 0.01]:
            for ponderation in [None, "balanced"]:
                modele = construire_modele_svm(valeur_C, valeur_gamma, ponderation)
                nom_de_l_essai = f"svm_C{valeur_C}_gamma{valeur_gamma}_cw{ponderation}"
                hyperparametres = {"kernel": "rbf", "C": valeur_C, "gamma": valeur_gamma,"class_weight": ponderation}
                metriques = enregistrer_un_essai(modele, nom_de_l_essai, hyperparametres)
                print(nom_de_l_essai, "PR-AUC =", round(metriques["cv_pr_auc_mean"], 3)," et rappel =", round(metriques["cv_recall_mean"], 3))

svm_C0.1_gammascale_cwNone PR-AUC = 0.653  et rappel = 0.184
svm_C0.1_gammascale_cwbalanced PR-AUC = 0.628  et rappel = 0.764


c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Us

svm_C0.1_gamma0.01_cwNone PR-AUC = 0.645  et rappel = 0.0
svm_C0.1_gamma0.01_cwbalanced PR-AUC = 0.521  et rappel = 0.758
svm_C1_gammascale_cwNone PR-AUC = 0.667  et rappel = 0.381
svm_C1_gammascale_cwbalanced PR-AUC = 0.653  et rappel = 0.748
svm_C1_gamma0.01_cwNone PR-AUC = 0.649  et rappel = 0.135
svm_C1_gamma0.01_cwbalanced PR-AUC = 0.627  et rappel = 0.758
svm_C10_gammascale_cwNone PR-AUC = 0.65  et rappel = 0.449
svm_C10_gammascale_cwbalanced PR-AUC = 0.613  et rappel = 0.707
svm_C10_gamma0.01_cwNone PR-AUC = 0.652  et rappel = 0.323
svm_C10_gamma0.01_cwbalanced PR-AUC = 0.661  et rappel = 0.761


In [10]:
chemin_du_fichier_csv = exporter_runs("churn_bancaire", "roman")
print(chemin_du_fichier_csv)

C:\Users\33698\Desktop\Cours\Master\M2 DS SIAD\projet_mlops\mlops_roman_delbare\reports\runs_roman.csv


En analysant le csv de tracking on voit que : 
La meilleure config semble être  C = 1, gamma = "scale" et sans class_weight car ca nous donne le "meilleur mix" : PR-AUC = 0,667 , ROC-AUC = 0,832, accuracy = 0,855, précision = 0,804, recall  = 0,381, F1 = 0,517 (NB : précision, rappel et F1 mesurés au seuil natif du SVM).
Le SVM est légèrement moins bon que le meilleur RF que j'ai testé hier et je pense que optimiser le seuil de décision sera plus simple avec la RF que sur un modèle type SVM donc pour l'instant la RF reste en tête de mon classement. 